# LangChain Financial PDF Agent → Microsoft Foundry Hosted Agent

Simple starter notebook: financial PDF RAG + LangChain/LangGraph agent, package as a container for Microsoft Foundry Hosted Agents, and view observability in Foundry/Application Insights.

## Architecture
```text
Financial PDF -> PDF loader/chunker -> embeddings/vector search
                                      |
User question -> LangGraph ReAct agent -> retrieval tool + calculator tool
                                      |
Local validation -> container image -> Microsoft Foundry Hosted Agent
                                      |
OpenTelemetry -> Foundry Traces + Azure Monitor Application Insights
```

In [ ]:
%pip install -U langchain langchain-openai langchain-community langgraph pypdf faiss-cpu pandas numpy matplotlib reportlab azure-identity azure-ai-projects azure-monitor-query opentelemetry-sdk azure-core-tracing-opentelemetry

## 1. Environment variables

In [ ]:
import os
from getpass import getpass
os.environ.setdefault("AZURE_OPENAI_ENDPOINT", "https://<your-aoai-resource>.openai.azure.com/")
os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
os.environ.setdefault("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4.1")
os.environ.setdefault("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
os.environ.setdefault("AZURE_AI_PROJECT_ENDPOINT", "https://<foundry-resource>.services.ai.azure.com/api/projects/<project-name>")
os.environ.setdefault("AZURE_SUBSCRIPTION_ID", "<subscription-id>")
os.environ.setdefault("AZURE_RESOURCE_GROUP", "<resource-group>")
if "AZURE_OPENAI_API_KEY" not in os.environ:
    os.environ["AZURE_OPENAI_API_KEY"] = getpass("Azure OpenAI API key: ")
for k in ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_VERSION", "AZURE_OPENAI_CHAT_DEPLOYMENT", "AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "AZURE_AI_PROJECT_ENDPOINT"]:
    print(k, "=", os.environ.get(k))

## 2. Create a sample financial PDF

In [ ]:
from pathlib import Path
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "sample_portfolio_statement.pdf"
sample_text = """
Jay Sample Household Portfolio Statement - Q1 2026
Total Portfolio Value: $1,250,000
Holdings:
- MSFT: $420,000, Technology, US Large Cap Equity
- VTI: $300,000, Broad US Equity ETF
- VXUS: $125,000, International Equity ETF
- BND: $150,000, US Aggregate Bond ETF
- SGOV: $75,000, Treasury Bills / Cash Equivalent
- NVDA: $95,000, Technology, US Large Cap Equity
- AMZN: $85,000, Consumer / Technology, US Large Cap Equity
Investment Objectives:
- Long-term wealth accumulation
- Preserve flexibility for education funding
- Avoid more than 35% single-stock exposure
- Maintain 15% to 25% defensive assets such as bonds, T-bills, or cash equivalents
Risk Notes:
- Technology exposure is high because MSFT, NVDA, and AMZN are large positions.
- Bond and T-bill allocation provides some downside protection.
- International diversification is modest.
Recent Cash Flows:
- Monthly contribution: $7,500
- 529 contribution: $350/month
- Planned annual bonus investment: $40,000
""".strip()
pdf = canvas.Canvas(str(PDF_PATH), pagesize=letter)
width, height = letter; x, y = 72, height - 72
for line in sample_text.splitlines():
    if y < 72:
        pdf.showPage(); y = height - 72
    pdf.drawString(x, y, line[:110]); y -= 16
pdf.save(); print("Created", PDF_PATH.resolve())

## 3. Load, chunk, embed, and create vector store

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
loader = PyPDFLoader(str(PDF_PATH)); docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120).split_documents(docs)
embeddings = AzureOpenAIEmbeddings(azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], api_key=os.environ["AZURE_OPENAI_API_KEY"], api_version=os.environ["AZURE_OPENAI_API_VERSION"], model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"])
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Loaded {len(docs)} page(s), created {len(chunks)} chunk(s)")

## 4. Define LangChain tools

In [ ]:
import re, pandas as pd
from langchain_core.tools import tool
@tool
def retrieve_portfolio_context(question: str) -> str:
    """Retrieve relevant context from the portfolio PDF for a user question."""
    return "\n\n".join([d.page_content for d in retriever.invoke(question)])
@tool
def calculate_portfolio_allocations(raw_text: str = "") -> str:
    """Calculate allocation, concentration, and risk flags from portfolio text."""
    text = raw_text or sample_text
    rows=[]
    for ticker, value, desc in re.findall(r"- ([A-Z]+): \$([0-9,]+), ([^\n]+)", text):
        rows.append({"Ticker": ticker, "Value": float(value.replace(',', '')), "Description": desc})
    df=pd.DataFrame(rows)
    if df.empty: return "No holdings extracted."
    total=df["Value"].sum(); df["Allocation"]=df["Value"]/total
    defensive=df[df["Ticker"].isin(["BND","SGOV"] )]["Value"].sum()/total
    single_stock=df[df["Ticker"].isin(["MSFT","NVDA","AMZN"] )]["Value"].sum()/total
    largest=df.sort_values("Value", ascending=False).iloc[0]
    return str({"total_value":round(total,2), "largest_holding":largest["Ticker"], "largest_holding_allocation_pct":round(largest["Allocation"]*100,2), "single_stock_allocation_pct":round(single_stock*100,2), "defensive_allocation_pct":round(defensive*100,2), "allocation_table":df.assign(Allocation=lambda x:(x["Allocation"]*100).round(2)).to_dict(orient="records")})
print(calculate_portfolio_allocations.invoke({}))

## 5. Build and test local LangGraph ReAct agent

In [ ]:
from langchain_openai import AzureChatOpenAI
from langgraph.prebuilt import create_react_agent
llm = AzureChatOpenAI(azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], api_key=os.environ["AZURE_OPENAI_API_KEY"], api_version=os.environ["AZURE_OPENAI_API_VERSION"], azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"], temperature=0)
system_prompt = """You are a financial portfolio analysis assistant. Use the PDF retrieval tool for statement facts and the calculator tool for allocations. Do not provide personalized investment advice. Return: Summary, Evidence from PDF, Calculations, Risks, Next questions."""
agent = create_react_agent(model=llm, tools=[retrieve_portfolio_context, calculate_portfolio_allocations], prompt=system_prompt)
response = agent.invoke({"messages": [("user", "Analyze this portfolio. Which risks need attention, and is it aligned to the stated objectives?")]})
print(response["messages"][-1].content)

## 6. Visualize allocation locally

In [ ]:
import ast, matplotlib.pyplot as plt
calc=ast.literal_eval(calculate_portfolio_allocations.invoke({}))
df_alloc=pd.DataFrame(calc["allocation_table"])
plt.figure(figsize=(8,4)); plt.bar(df_alloc["Ticker"], df_alloc["Allocation"])
plt.title("Portfolio allocation by holding"); plt.ylabel("Allocation %"); plt.xlabel("Holding"); plt.show()

# Hosted Agent packaging

These cells create a minimal containerized app. In production, copy the business logic into the Azure Samples repo Stage 4 hosted-agent file and preserve its hosted-agent protocol adapter / `azd` infra.

In [ ]:
from pathlib import Path
import shutil
HOSTED_DIR=Path("hosted_portfolio_agent"); (HOSTED_DIR/"data").mkdir(parents=True, exist_ok=True)
shutil.copy(PDF_PATH, HOSTED_DIR/"data"/PDF_PATH.name)
print("Created", HOSTED_DIR.resolve())

In [ ]:
%%writefile hosted_portfolio_agent/requirements.txt
fastapi
uvicorn[standard]
langchain
langchain-openai
langchain-community
langgraph
pypdf
faiss-cpu
pandas
numpy
azure-identity
azure-core-tracing-opentelemetry
opentelemetry-sdk

In [ ]:
%%writefile hosted_portfolio_agent/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile hosted_portfolio_agent/app.py
import os, re
from pathlib import Path
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
try:
    from azure.core.settings import settings
    from azure.core.tracing.ext.opentelemetry_span import OpenTelemetrySpan
    settings.tracing_implementation = OpenTelemetrySpan
except Exception:
    pass
PDF_PATH = Path(__file__).parent / 'data' / 'sample_portfolio_statement.pdf'
app = FastAPI(title='Portfolio LangChain Hosted Agent')
class ChatRequest(BaseModel):
    message: str
def build_agent():
    docs = PyPDFLoader(str(PDF_PATH)).load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120).split_documents(docs)
    embeddings = AzureOpenAIEmbeddings(azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'], api_key=os.environ.get('AZURE_OPENAI_API_KEY'), api_version=os.environ.get('AZURE_OPENAI_API_VERSION','2024-12-01-preview'), model=os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT','text-embedding-3-large'))
    retriever = FAISS.from_documents(chunks, embeddings).as_retriever(search_kwargs={'k':4})
    full_text = '\n'.join(d.page_content for d in docs)
    @tool
    def retrieve_portfolio_context(question: str) -> str:
        '''Retrieve relevant context from the portfolio PDF for a user question.'''
        return '\n\n'.join(d.page_content for d in retriever.invoke(question))
    @tool
    def calculate_portfolio_allocations(raw_text: str = '') -> str:
        '''Calculate allocation, concentration, and risk flags from extracted PDF text.'''
        text = raw_text or full_text
        rows=[]
        for ticker, value, desc in re.findall(r'- ([A-Z]+): \$([0-9,]+), ([^\n]+)', text):
            rows.append({'Ticker':ticker, 'Value':float(value.replace(',', '')), 'Description':desc})
        df=pd.DataFrame(rows)
        if df.empty: return 'No holdings extracted. Use Document Intelligence or a cleaner source PDF.'
        total=df['Value'].sum(); df['Allocation']=df['Value']/total
        defensive=df[df['Ticker'].isin(['BND','SGOV'])]['Value'].sum()/total
        single_stock=df[df['Ticker'].isin(['MSFT','NVDA','AMZN'])]['Value'].sum()/total
        largest=df.sort_values('Value', ascending=False).iloc[0]
        return str({'total_value':round(total,2), 'largest_holding':largest['Ticker'], 'largest_holding_allocation_pct':round(largest['Allocation']*100,2), 'single_stock_allocation_pct':round(single_stock*100,2), 'defensive_allocation_pct':round(defensive*100,2), 'allocation_table':df.assign(Allocation=lambda x:(x['Allocation']*100).round(2)).to_dict(orient='records')})
    llm=AzureChatOpenAI(azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'], api_key=os.environ.get('AZURE_OPENAI_API_KEY'), api_version=os.environ.get('AZURE_OPENAI_API_VERSION','2024-12-01-preview'), azure_deployment=os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT'], temperature=0)
    prompt='You are a financial portfolio analysis assistant. Use tools for PDF facts and calculations. Do not provide personalized investment advice.'
    return create_react_agent(model=llm, tools=[retrieve_portfolio_context, calculate_portfolio_allocations], prompt=prompt)
AGENT=None
@app.on_event('startup')
def startup():
    global AGENT; AGENT=build_agent()
@app.get('/health')
def health(): return {'status':'ok'}
@app.post('/chat')
def chat(req: ChatRequest):
    result=AGENT.invoke({'messages':[('user', req.message)]})
    return {'response':result['messages'][-1].content}

## 7. Test container locally

```bash
cd hosted_portfolio_agent

docker build -t portfolio-langchain-agent:local .

docker run --rm -p 8000:8000 \
  -e AZURE_OPENAI_ENDPOINT="$AZURE_OPENAI_ENDPOINT" \
  -e AZURE_OPENAI_API_KEY="$AZURE_OPENAI_API_KEY" \
  -e AZURE_OPENAI_API_VERSION="$AZURE_OPENAI_API_VERSION" \
  -e AZURE_OPENAI_CHAT_DEPLOYMENT="$AZURE_OPENAI_CHAT_DEPLOYMENT" \
  -e AZURE_OPENAI_EMBEDDING_DEPLOYMENT="$AZURE_OPENAI_EMBEDDING_DEPLOYMENT" \
  portfolio-langchain-agent:local

curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message":"Which portfolio risks need attention this quarter?"}'
```

## 8. Deploy to Microsoft Foundry Hosted Agent

Recommended path with the sample repo:

```bash
git clone https://github.com/Azure-Samples/foundry-hosted-langchain-demos.git
cd foundry-hosted-langchain-demos
azd auth login
azd up
```

Then replace the Stage 4 agent business logic with `hosted_portfolio_agent/app.py`, while preserving the sample repo Hosted Agent protocol wiring.

Direct container lifecycle example:

```bash
az login
azd auth login
ACR_NAME=<your-acr-name>
IMAGE_NAME=portfolio-langchain-agent
IMAGE_TAG=v1
az acr login --name $ACR_NAME
az acr build --registry $ACR_NAME --image $IMAGE_NAME:$IMAGE_TAG ./hosted_portfolio_agent
```

The Hosted Agent lifecycle is: build/push image to ACR → create Foundry hosted-agent version → poll until active → invoke the dedicated endpoint.

## 9. Enable Foundry observability

In Foundry portal: open your project → **Agents → Traces** → **Connect** Application Insights → invoke the hosted agent → review traces in Foundry and Application Insights Transaction search / Performance.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient
from datetime import timedelta
import pandas as pd
LOG_ANALYTICS_WORKSPACE_ID = os.environ.get("LOG_ANALYTICS_WORKSPACE_ID", "<workspace-id>")
kql = """
requests
| where timestamp > ago(24h)
| order by timestamp desc
| project timestamp, name, resultCode, success, duration, operation_Id
| take 20
"""
if LOG_ANALYTICS_WORKSPACE_ID.startswith("<"):
    print("Set LOG_ANALYTICS_WORKSPACE_ID to query traces.")
else:
    client = LogsQueryClient(DefaultAzureCredential())
    result = client.query_workspace(LOG_ANALYTICS_WORKSPACE_ID, kql, timespan=timedelta(days=1))
    for table in result.tables:
        display(pd.DataFrame(table.rows, columns=table.columns))

## Next production upgrades
- Replace local FAISS with Azure AI Search or Foundry Toolbox retrieval.
- Use Document Intelligence / Content Understanding for robust PDF table extraction.
- Add trace attributes like `portfolio_id`, `tool_name`, `retrieval_k`, `latency_ms`, `risk_category`.
- Add evaluations: faithfulness to PDF, calculation accuracy, no-personal-advice guardrail, answer completeness.